# Scenario Calendar


In [1]:
import numpy as np

## Inputs

In [2]:
ASSETS = ["cabins", "buggies", "barriers", "fencing", "flooring"]

# Months in season order (Oct..Sep) and their lengths, used to place new events.
MONTHS = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MONTH_LEN = [31, 30, 31, 31, 28, 31, 30, 31, 30, 31, 31, 30]
MONTH_START = np.cumsum([0] + MONTH_LEN[:-1])  # day offset of each month's 1st

### Y1 Event Calendar

In [3]:
# (name, hire start, hire end, cabin, buggies, barriers, fencing, flooring)
events = [
    ('Saadiyat Nights',1,157,10,10,0,0,280),
    ('BRED X',22,43,8,0,0,0,0),
    ('MOTN DHA',34,56,8,6,350,550,0),
    ('No Art',36,57,24,6,0,0,200),
    ('Tanweer Festival',44,56,0,8,900,50,200),
    ('MOTN AAN',40,66,18,6,350,550,0),
    ('Keinemusik',57,69,29,8,714,1180,178),
    ('MOTN AUH',50,102,20,10,700,1100,200),
    ('Yas Winterfest',56,88,6,4,644,760,60),
    ('Animenia',118,140,8,4,288,192,0),
    ('DAZ',170,200,15,4,775,225,60),
    ('BRED',196,210,22,6,1500,500,0),
]
start = np.array([r[1] for r in events])
end = np.array([r[2] for r in events])
qty = np.array([r[3:] for r in events], dtype=float)

### Seasonal Weights

In [4]:
month_idx = np.searchsorted(MONTH_START, start, side="right") - 1
months = np.array(MONTHS)[month_idx]

counts = np.bincount(month_idx, minlength=len(MONTHS))
y1_weights = counts / counts.sum()

for month, count, weight in zip(MONTHS, counts, y1_weights):
    print(f"{month:>2}: {count:2d} events ({weight:6.1%})")

10:  2 events ( 16.7%)
11:  7 events ( 58.3%)
12:  0 events (  0.0%)
 1:  1 events (  8.3%)
 2:  0 events (  0.0%)
 3:  1 events (  8.3%)
 4:  1 events (  8.3%)
 5:  0 events (  0.0%)
 6:  0 events (  0.0%)
 7:  0 events (  0.0%)
 8:  0 events (  0.0%)
 9:  0 events (  0.0%)


In [5]:
# Seasonal weights for placing NEW events (Oct..Sep) — concentrated Oct–Jan.
weights = np.array([4, 14, 8, 6, 4, 2, 1, 0, 0, 0, 0, 0])
selected_weights = weights / weights.sum()

for month, weight in zip(MONTHS, selected_weights):
    print(f"{month:>2}: {weight:6.1%}")

10:  10.3%
11:  35.9%
12:  20.5%
 1:  15.4%
 2:  10.3%
 3:   5.1%
 4:   2.6%
 5:   0.0%
 6:   0.0%
 7:   0.0%
 8:   0.0%
 9:   0.0%


### Event Duration & Asset Archetype

In [6]:
dur = end - start + 1
mean_dur = np.mean(dur, axis=0)
median_dur = np.median(dur, axis=0)

mean_qty = np.mean(qty, axis=0)
median_qty = np.median(qty, axis=0)

print(f"Mean Dur.: {mean_dur:.0f}days, Median Dur.: {median_dur:.0f}days")
print("")

width = max(len(asset) for asset in ASSETS)
print(f"{'Asset':<{width}}  {'Mean':>10}  {'Median':>10}")
print(f"{'-' * width}  {'-' * 10}  {'-' * 10}")
for asset, mean, median in zip(ASSETS, mean_qty, median_qty):
    print(
        f"{asset:<{width}}  "
        f"{mean:10.1f}  "
        f"{median:10.1f}"
    )

Mean Dur.: 36days, Median Dur.: 23days

Asset           Mean      Median
--------  ----------  ----------
cabins          14.0        12.5
buggies          6.0         6.0
barriers       518.4       497.0
fencing        425.6       362.5
flooring        98.2        60.0


In [7]:
# (duration, ["cabins", "buggies", "barriers", "fencing", "flooring"])
archetype = (25, np.array([13, 6, 500, 400, 100], dtype=float))

## Simulation

### Peak Simulation

In [8]:
def simulate_peaks(start, end, qty, win_rate, n_trials, rng, horizon):
    """Coin-flip each event n_trials times; for the events that 'sign', walk the
    calendar and record the highest concurrent demand. Returns the peak for
    every trial: an array of shape (n_trials, n_assets).

      - on_hire[i, d]  = 1 if event i is on hire on day d
      - signed[t, i]   = 1 if event i won in trial t  (the coin flip)
      - demand[t, d]   = sum over signed events of their qty on that day
      - peak[t]        = max over days
    """
    n_events = len(start)

    # on_hire[i, d] : is event i on hire on day d?   (n_events x horizon)
    days = np.arange(horizon)
    on_hire = (days[None, :] >= start[:, None]) & (days[None, :] <= end[:, None])
    on_hire = on_hire.astype(np.float32)

    # signed[t, i] : did event i sign in trial t?   (n_trials x n_events)
    signed = (rng.random((n_trials, n_events)) < win_rate).astype(np.float32)

    # Per asset: daily demand = signed events' units summed by day; peak = max day.
    peaks = np.empty((n_trials, len(ASSETS)), dtype=np.float32)
    for a in range(len(ASSETS)):
        units_per_day = on_hire * qty[:, a][:, None]  # (n_events x horizon)
        demand = signed @ units_per_day  # (n_trials x horizon)
        peaks[:, a] = demand.max(axis=1)
    return peaks